In [ ]:
%py
# Test script to validate masking of last four digits of invoice_number in purgo_playground.d_product_revenue_clone

import unittest
from pyspark.sql.types import StructType, StructField, StringType, LongType, DateType, DoubleType, TimestampType
from pyspark.sql.functions import col, udf
import datetime
import time

# Commented out SparkSession initialization as spark is already available in Databricks
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("InvoiceNumberMaskingTest").getOrCreate()

# Define the masking function
def mask_invoice_number(invoice):
    if invoice is None:
        return None
    invoice_str = str(invoice)
    if len(invoice_str) <= 4:
        return '*' * len(invoice_str)
    else:
        return invoice_str[:-4] + '****'

mask_udf = udf(mask_invoice_number, StringType())

class TestInvoiceNumberMasking(unittest.TestCase):
    # Setup and Configuration
    def setUp(self):
        # Drop the clone table if it exists
        try:
            spark.sql("DROP TABLE IF EXISTS purgo_playground.d_product_revenue_clone")
        except Exception as e:
            self.fail(f"Failed to drop clone table: {e}")
        
        # Create the clone table from the source table
        try:
            spark.sql("""
                CREATE TABLE purgo_playground.d_product_revenue_clone AS
                SELECT * FROM purgo_playground.d_product_revenue
            """)
        except Exception as e:
            self.fail(f"Failed to create clone table: {e}")

    def tearDown(self):
        # Cleanup operations
        try:
            spark.sql("DROP TABLE IF EXISTS purgo_playground.d_product_revenue_clone")
        except Exception as e:
            print(f"Failed to drop clone table during teardown: {e}")

    # Unit Tests for Data Transformation
    def test_mask_normal_invoice_numbers(self):
        # Define test cases with original and expected masked invoice_numbers
        test_cases = [
            ("1234234534", "123423****"),
            ("9876543210", "987654****"),
            ("555566667777", "55556666****"),
            ("1000", "****")
        ]
        for original, expected in test_cases:
            # Update the invoice_number with masked value
            try:
                spark.sql(f"""
                    UPDATE purgo_playground.d_product_revenue_clone
                    SET invoice_number = '{mask_invoice_number(original)}'
                    WHERE invoice_number = {original}
                """)
            except Exception as e:
                self.fail(f"Failed to update invoice_number {original}: {e}")
            
            # Validate the masked value
            result = spark.sql(f"""
                SELECT invoice_number FROM purgo_playground.d_product_revenue_clone
                WHERE invoice_number = '{expected}'
            """).collect()
            self.assertEqual(len(result), 1, f"Masked invoice_number {expected} not found.")

    def test_mask_invoice_less_than_four_digits(self):
        # Test masking for invoice_number with less than four digits
        original = "123"
        expected = "***"
        try:
            spark.sql(f"""
                UPDATE purgo_playground.d_product_revenue_clone
                SET invoice_number = '{mask_invoice_number(original)}'
                WHERE invoice_number = {original}
            """)
        except Exception as e:
            self.fail(f"Failed to update invoice_number {original}: {e}")
        
        result = spark.sql(f"""
            SELECT invoice_number FROM purgo_playground.d_product_revenue_clone
            WHERE invoice_number = '{expected}'
        """).collect()
        self.assertEqual(len(result), 1, f"Masked invoice_number {expected} not found.")

    def test_preserve_other_columns(self):
        # Ensure other columns remain unchanged after masking
        product_id = 1
        original_data = spark.sql(f"""
            SELECT * FROM purgo_playground.d_product_revenue_clone
            WHERE product_id = {product_id}
        """).collect()[0]
        try:
            spark.sql(f"""
                UPDATE purgo_playground.d_product_revenue_clone
                SET invoice_number = '{mask_invoice_number(original_data.invoice_number)}'
                WHERE product_id = {product_id}
            """)
        except Exception as e:
            self.fail(f"Failed to mask invoice_number for product_id {product_id}: {e}")
        
        updated_data = spark.sql(f"""
            SELECT * FROM purgo_playground.d_product_revenue_clone
            WHERE product_id = {product_id}
        """).collect()[0]
        
        # Validate other columns
        self.assertEqual(updated_data['product_id'], original_data['product_id'])
        self.assertEqual(updated_data['product_name'], original_data['product_name'])
        self.assertEqual(updated_data['revenue'], original_data['revenue'])
        self.assertEqual(updated_data['country'], original_data['country'])

    # Integration Tests for End-to-End Flow
    def test_create_clone_table_failure(self):
        # Drop source table and attempt to create clone to trigger failure
        try:
            spark.sql("DROP TABLE IF EXISTS purgo_playground.d_product_revenue")
            with self.assertRaises(Exception):
                spark.sql("""
                    CREATE TABLE purgo_playground.d_product_revenue_clone AS
                    SELECT * FROM purgo_playground.d_product_revenue
                """)
        except Exception:
            pass  # Expected exception

    def test_missing_invoice_number_column(self):
        # Drop invoice_number column and attempt masking to trigger failure
        try:
            spark.sql("ALTER TABLE purgo_playground.d_product_revenue_clone DROP COLUMN invoice_number")
            with self.assertRaises(Exception):
                df = spark.table("purgo_playground.d_product_revenue_clone")
                df.withColumn("invoice_number", mask_udf(col("invoice_number"))).write.mode("overwrite").saveAsTable("purgo_playground.d_product_revenue_clone")
        except Exception:
            pass  # Expected exception

    # Data Type Validation Tests
    def test_data_type_after_masking(self):
        # Apply masking and validate data type of invoice_number
        try:
            df = spark.table("purgo_playground.d_product_revenue_clone")
            masked_df = df.withColumn("invoice_number", mask_udf(col("invoice_number")))
            masked_df.write.mode("overwrite").saveAsTable("purgo_playground.d_product_revenue_clone")
        except Exception as e:
            self.fail(f"Failed to apply masking: {e}")
        
        schema = spark.table("purgo_playground.d_product_revenue_clone").schema
        field = schema["invoice_number"]
        self.assertEqual(field.dataType, StringType(), "invoice_number is not of type StringType after masking.")

    def test_masked_invoice_format(self):
        # Ensure all masked invoice_numbers have last four characters replaced with '*'
        df = spark.table("purgo_playground.d_product_revenue_clone")
        masked_df = df.filter(col("invoice_number").rlike(".*\\*\\*\\*\\*$"))
        self.assertEqual(masked_df.count(), df.count(), "Not all invoice_numbers are properly masked.")

    # Performance Tests
    def test_performance_large_volume(self):
        # Test masking performance with 10 million records within 30 minutes
        try:
            # Assuming the clone table has 10 million records
            df = spark.table("purgo_playground.d_product_revenue_clone")
            start_time = time.time()
            masked_df = df.withColumn("invoice_number", mask_udf(col("invoice_number")))
            masked_df.write.mode("overwrite").saveAsTable("purgo_playground.d_product_revenue_clone")
            end_time = time.time()
            duration = end_time - start_time
            self.assertTrue(duration < 1800, f"Masking took longer than 30 minutes: {duration} seconds")
        except Exception as e:
            self.fail(f"Performance test failed: {e}")

    # Data Quality Validation Tests
    def test_no_original_data_retained(self):
        # Ensure no original invoice_number data is retrievable after masking
        df = spark.table("purgo_playground.d_product_revenue_clone")
        original_data = df.filter(col("invoice_number").rlike("^[0-9]+$"))
        self.assertEqual(original_data.count(), 0, "Original invoice_number data is still present after masking.")

    def test_null_invoice_number_handling(self):
        # Validate handling of null invoice_number values
        try:
            df_before = spark.table("purgo_playground.d_product_revenue_clone").filter(col("invoice_number").isNull())
            masked_df = spark.table("purgo_playground.d_product_revenue_clone").withColumn("invoice_number", mask_udf(col("invoice_number")))
            masked_df.write.mode("overwrite").saveAsTable("purgo_playground.d_product_revenue_clone")
            df_after = spark.table("purgo_playground.d_product_revenue_clone").filter(col("invoice_number").isNull())
            self.assertEqual(df_before.count(), df_after.count(), "Rows with null invoice_number were altered.")
        except Exception as e:
            self.fail(f"Null handling test failed: {e}")

# Run the tests
if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

# Commented out spark.stop() to prevent issues in Databricks
# spark.stop()